In [ ]:
import os
import pandas as pd 

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
    execute_sql,
)

from utils.synthetic.pipeline.rebuild_comparison import build_rebuild_comparison_stage

from utils.core.env_helpers import (
    env_float, 
    env_int, 
    env_str
)

# Show every column regardless of count
pd.set_option('display.max_columns', None)

# Prevent columns from wrapping to a new line
pd.set_option('display.width', 1000)

# Optional: Prevent cell content inside columns from cutting off
pd.set_option('display.max_colwidth', None)

In [14]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

FLOAT_TOLERANCE = env_float(
    "SYNTHETIC_REBUILD_COMPARISON_FLOAT_TOLERANCE",
    1e-9,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

PREMELT_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

REBUILT_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

TARGET_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILD_COMPARISON_TABLE",
    "synthetic_sensor_rebuild_comparison_stage",
)

print("Rebuild comparison config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("premelt table:", PREMELT_SOURCE_TABLE_NAME)
print("rebuilt table:", REBUILT_DESTINATION_TABLE_NAME)
print("target table:", TARGET_TABLE_NAME)

Rebuild comparison config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
premelt table: synthetic_observations_premelt_stage
rebuilt table: synthetic_sensor_observations_rebuilt_stage
target table: synthetic_sensor_rebuild_comparison_stage


In [15]:
engine = get_engine_from_env()

----

In [17]:
comparison_result = build_rebuild_comparison_stage(
    engine=engine,
    schema=SCHEMA,
    premelt_table=PREMELT_SOURCE_TABLE_NAME,
    rebuilt_table=REBUILT_DESTINATION_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    float_tolerance=FLOAT_TOLERANCE,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(comparison_result)

[obs-window] 1 | observation_index 1 to 2,500
[obs-window] 2 | observation_index 2,501 to 5,000
[obs-window] 3 | observation_index 5,001 to 7,500
[obs-window] 4 | observation_index 7,501 to 10,000
[obs-window] 5 | observation_index 10,001 to 12,500
[obs-window] 6 | observation_index 12,501 to 15,000
[obs-window] 7 | observation_index 15,001 to 17,500
[obs-window] 8 | observation_index 17,501 to 20,000
[obs-window] 9 | observation_index 20,001 to 22,500
[obs-window] 10 | observation_index 22,501 to 25,000
[obs-window] 11 | observation_index 25,001 to 27,500
[obs-window] 12 | observation_index 27,501 to 30,000
[obs-window] 13 | observation_index 30,001 to 32,500
[obs-window] 14 | observation_index 32,501 to 35,000
[obs-window] 15 | observation_index 35,001 to 37,500
[obs-window] 16 | observation_index 37,501 to 40,000
[obs-window] 17 | observation_index 40,001 to 42,500
[obs-window] 18 | observation_index 42,501 to 45,000
[obs-window] 19 | observation_index 45,001 to 47,500
[obs-window] 

{'status': 'built',
 'comparison_rows': 225000,
 'target_table': 'synthetic_sensor_rebuild_comparison_stage'}

----

In [19]:
summary_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS comparison_row_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = TRUE) AS all_match_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = FALSE) AS mismatch_count,
        MAX(comparison_mismatch_count) AS max_mismatch_count
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    
    """,
)

display(summary_dataframe)

,comparison_row_count,all_match_count,mismatch_count,max_mismatch_count
0,225000,225000,0,0


----

In [20]:
mismatch_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        comparison_mismatch_count,
        comparison_notes,
        exists_in_original,
        exists_in_rebuilt,
        rebuilt_rebuild_sensor_count,
        rebuilt_rebuild_is_complete
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
      AND comparison_all_fields_match = FALSE
    ORDER BY observation_index
    LIMIT 50
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(mismatch_dataframe)

,dataset_id,run_id,asset_id,observation_index,comparison_mismatch_count,comparison_notes,exists_in_original,exists_in_rebuilt,rebuilt_rebuild_sensor_count,rebuilt_rebuild_is_complete


-----

In [21]:
comparison_columns_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": TARGET_TABLE_NAME,
    },
)

match_columns = [
    column
    for column in comparison_columns_dataframe["column_name"].tolist()
    if column.startswith("match")
]

print("Match columns found:", len(match_columns))
display(pd.DataFrame({"match_column": match_columns}).head(100))

Match columns found: 58


,match_column
0,match_generated_row_id
1,match_stream_state
2,match_phase
3,match_meta_episode_id
4,match_meta_primary_fault_type
5,match_meta_magnitude
6,match_sensor_00
7,match_sensor_01
8,match_sensor_02
9,match_sensor_03


In [22]:
match_failure_sql_parts = [
    f"""
    SELECT
        '{column}' AS match_column,
        COUNT(*) FILTER (WHERE "{column}" = FALSE) AS failed_count,
        COUNT(*) FILTER (WHERE "{column}" = TRUE) AS passed_count
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """
    for column in match_columns
]

match_failure_summary_dataframe = read_sql_dataframe(
    engine,
    "\nUNION ALL\n".join(match_failure_sql_parts)
    + "\nORDER BY failed_count DESC, match_column",
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(match_failure_summary_dataframe)

,match_column,failed_count,passed_count
0,match_generated_row_id,0,225000
1,match_meta_episode_id,0,225000
2,match_meta_magnitude,0,225000
3,match_meta_primary_fault_type,0,225000
4,match_phase,0,225000
5,match_sensor_00,0,225000
6,match_sensor_01,0,225000
7,match_sensor_02,0,225000
8,match_sensor_03,0,225000
9,match_sensor_04,0,225000


In [23]:
episode_id_mismatch_sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        original_meta_episode_id,
        rebuilt_meta_episode_id,
        match_meta_episode_id
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
      AND match_meta_episode_id = FALSE
    ORDER BY observation_index
    LIMIT 50
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(episode_id_mismatch_sample_dataframe)

,dataset_id,run_id,asset_id,observation_index,original_meta_episode_id,rebuilt_meta_episode_id,match_meta_episode_id


---

In [24]:
observation_to_check = 1

detail_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE observation_index = {int(observation_to_check)}
    """,
)

display(detail_dataframe)

,dataset_id,run_id,asset_id,observation_index,exists_in_original,exists_in_rebuilt,comparison_mismatch_count,comparison_all_fields_match,comparison_notes,comparison_completed_at,rebuilt_rebuild_sensor_count,rebuilt_rebuild_is_complete,original_generated_row_id,original_batch_id,original_row_in_batch,original_global_cycle_id,original_stream_state,original_phase,original_created_at,original_meta_episode_id,original_meta_primary_fault_type,original_meta_magnitude,original_is_telemetry_event,original_telemetry_event_type,original_producer_send_attempt,original_meta_is_fault_episode,original_meta_phase_truth,original_meta_primary_sensor,original_meta_primary_magnitude,original_meta_primary_sensor_abs_zscore,original_meta_primary_sensor_threshold_crossed,original_meta_fault_onset_truth_flag,original_meta_fault_onset_truth_row,original_meta_failure_truth_flag,original_meta_failure_truth_row,original_meta_observable_onset_flag,original_meta_observable_onset_row,original_meta_buildup_progress,original_meta_rows_until_failure,original_meta_rows_since_truth_onset,original_meta_rows_since_observable_onset,original_sensor_00,original_sensor_01,original_sensor_02,original_sensor_03,original_sensor_04,original_sensor_05,original_sensor_06,original_sensor_07,original_sensor_08,original_sensor_09,original_sensor_10,original_sensor_11,original_sensor_12,original_sensor_13,original_sensor_14,original_sensor_15,original_sensor_16,original_sensor_17,original_sensor_18,original_sensor_19,original_sensor_20,original_sensor_21,original_sensor_22,original_sensor_23,original_sensor_24,original_sensor_25,original_sensor_26,original_sensor_27,original_sensor_28,original_sensor_29,original_sensor_30,original_sensor_31,original_sensor_32,original_sensor_33,original_sensor_34,original_sensor_35,original_sensor_36,original_sensor_37,original_sensor_38,original_sensor_39,original_sensor_40,original_sensor_41,original_sensor_42,original_sensor_43,original_sensor_44,original_sensor_45,original_sensor_46,original_sensor_47,original_sensor_48,original_sensor_49,original_sensor_50,original_sensor_51,rebuilt_observation_timestamp,rebuilt_generated_row_id,rebuilt_stream_state,rebuilt_phase,rebuilt_meta_episode_id,rebuilt_meta_primary_fault_type,rebuilt_meta_magnitude,rebuilt_rebuild_completed_at,rebuilt_rebuild_notes,rebuilt_sensor_00,rebuilt_sensor_01,rebuilt_sensor_02,rebuilt_sensor_03,rebuilt_sensor_04,rebuilt_sensor_05,rebuilt_sensor_06,rebuilt_sensor_07,rebuilt_sensor_08,rebuilt_sensor_09,rebuilt_sensor_10,rebuilt_sensor_11,rebuilt_sensor_12,rebuilt_sensor_13,rebuilt_sensor_14,rebuilt_sensor_15,rebuilt_sensor_16,rebuilt_sensor_17,rebuilt_sensor_18,rebuilt_sensor_19,rebuilt_sensor_20,rebuilt_sensor_21,rebuilt_sensor_22,rebuilt_sensor_23,rebuilt_sensor_24,rebuilt_sensor_25,rebuilt_sensor_26,rebuilt_sensor_27,rebuilt_sensor_28,rebuilt_sensor_29,rebuilt_sensor_30,rebuilt_sensor_31,rebuilt_sensor_32,rebuilt_sensor_33,rebuilt_sensor_34,rebuilt_sensor_35,rebuilt_sensor_36,rebuilt_sensor_37,rebuilt_sensor_38,rebuilt_sensor_39,rebuilt_sensor_40,rebuilt_sensor_41,rebuilt_sensor_42,rebuilt_sensor_43,rebuilt_sensor_44,rebuilt_sensor_45,rebuilt_sensor_46,rebuilt_sensor_47,rebuilt_sensor_48,rebuilt_sensor_49,rebuilt_sensor_50,rebuilt_sensor_51,match_generated_row_id,match_stream_state,match_phase,match_meta_episode_id,match_meta_primary_fault_type,match_meta_magnitude,match_sensor_00,match_sensor_01,match_sensor_02,match_sensor_03,match_sensor_04,match_sensor_05,match_sensor_06,match_sensor_07,match_sensor_08,match_sensor_09,match_sensor_10,match_sensor_11,match_sensor_12,match_sensor_13,match_sensor_14,match_sensor_15,match_sensor_16,match_sensor_17,match_sensor_18,match_sensor_19,match_sensor_20,match_sensor_21,match_sensor_22,match_sensor_23,match_sensor_24,match_sensor_25,match_sensor_26,match_sensor_27,match_sensor_28,match_sensor_29,match_sensor_30,match_sensor_31,match_sensor_32,match_sensor_33,match_sensor_34,match_sensor_35,match_sensor_36,match_sensor_37,match_sens

----

In [25]:
stage_10_final_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS comparison_row_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = TRUE) AS all_fields_match_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = FALSE) AS mismatch_row_count,
        COUNT(*) FILTER (WHERE exists_in_original = TRUE AND exists_in_rebuilt = FALSE) AS missing_from_rebuilt_count,
        COUNT(*) FILTER (WHERE exists_in_original = FALSE AND exists_in_rebuilt = TRUE) AS missing_from_original_count,
        MAX(comparison_mismatch_count) AS max_mismatch_count,
        (
            COUNT(*) = 225000
            AND COUNT(*) FILTER (WHERE comparison_all_fields_match = TRUE) = 225000
            AND COUNT(*) FILTER (WHERE comparison_all_fields_match = FALSE) = 0
            AND COUNT(*) FILTER (WHERE exists_in_original = TRUE AND exists_in_rebuilt = FALSE) = 0
            AND COUNT(*) FILTER (WHERE exists_in_original = FALSE AND exists_in_rebuilt = TRUE) = 0
        ) AS ready_for_stage_11
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_10_final_validation_dataframe)

,comparison_row_count,all_fields_match_count,mismatch_row_count,missing_from_rebuilt_count,missing_from_original_count,max_mismatch_count,ready_for_stage_11
0,225000,225000,0,0,0,0,True


In [26]:
import utils.synthetic.pipeline.rebuild_comparison as rebuild_comparison_module

print(rebuild_comparison_module.__file__)
print(rebuild_comparison_module._compare_scalar(0, "0"))
print(rebuild_comparison_module._compare_scalar(0.0, "0"))
print(rebuild_comparison_module._compare_scalar("0", "0"))

/workspace/utils/synthetic/pipeline/rebuild_comparison.py
True
True
True


In [ ]:
'''
execute_sql(
    engine,
    f"""
    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE
    """
)

print(f"Dropped {SCHEMA}.{TARGET_TABLE_NAME}")
'''

Dropped capstone.synthetic_sensor_rebuild_comparison_stage


In [27]:
# Dispose Engine
engine.dispose()